# Guided Project: Testing the Solver
**MATH 3515 · Module 13 · First segment of the Challenge Project *Predicting an Outbreak***

**Scenario.** A public-health office tracks a quantity that grows quickly at first and then levels off (people who have heard about a new vaccine clinic, a bacterial culture, the infected people in a simple epidemic). It uses the **logistic model**
$$x' = r\,x\left(1-\frac{x}{K}\right),\qquad x(0)=x_0,$$
and must compute it **step by step**, because the models it really cares about (the SIR model of the Challenge Project) have no formula. Before trusting a step-by-step method on those, the office tests it here, where the exact answer is known:
$$x(t)=\frac{K x_0}{x_0+(K-x_0)\,e^{-rt}}.$$
The office asks: **(1)** How accurate are Euler and RK4, and how does the accuracy improve when the step $h$ is halved? **(2)** How large can the step be before the method misbehaves?

**What you hand in** (due date in Blackboard): only the report `GP_Testing_Solver_Report.docx`, with **1. Results** (Part 6 prints the tables) and **2. Handwritten Computations** (photos of your Part 2 work on paper). The notebook is not collected, but keep it: the Challenge Project reuses it.

**Resources:** the Module 13 slides, including the backup slide *Predict the Safe Step* in the *Step Size and Stability* slides (Module 13); class notebooks 08 and 09.
**AI use: Level 2 (AI Collaboration).** You may use an assistant for code and for checking your reasoning; you are responsible for verifying everything against the slides, your hand work, and this notebook.

**What you type:** only your student ID (Part 0). Everything else is given: run each cell, look at the output, and write a short *Observation* (1–2 sentences: what you see, not why) where asked. Part 2 is done on paper first.

In [ ]:
# Setup: run this cell first
import hashlib, re
import numpy as np
import matplotlib.pyplot as plt

# short error messages: show only the last line of an error
try:
    get_ipython().run_line_magic("xmode", "Minimal")
except NameError:
    pass

# make unfilled blanks fail loudly (IPython predefines ___)
try:
    get_ipython().displayhook.cache_size = 0
    del ___
except NameError:
    pass

# Okabe-Ito colorblind-safe palette
OI = ["#0072B2", "#D55E00", "#009E73", "#CC79A7", "#E69F00", "#56B4E9", "#000000"]

def n_steps(T, h):
    n = round(T / h)
    if abs(n * h - T) > 1e-12:
        raise ValueError("T/h must be a whole number; choose h = 2**-k")
    return n

def euler(f, t0, x0, h, T):
    n = n_steps(T - t0, h)
    t, x = t0, x0
    for k in range(n):
        x = x + h * f(t, x)     # follow the tangent line
        t = t + h
    return x

def rk4(f, t0, x0, h, T):
    n = n_steps(T - t0, h)
    t, x = t0, x0
    for k in range(n):
        k1 = f(t, x)
        k2 = f(t + h / 2, x + h / 2 * k1)
        k3 = f(t + h / 2, x + h / 2 * k2)
        k4 = f(t + h, x + h * k3)
        x = x + h / 6 * (k1 + 2 * k2 + 2 * k3 + k4)
        t = t + h
    return x

print("Ready.")

## Part 0. Your personal parameters  ✎ YOUR TURN
Type your **Catawba student ID** between the quotes. It has **9 characters** and starts with `C203` (format: `C20312345`). The cell turns it into your own model:
- $r$, the growth rate (between 0.30 and 0.90 per day);
- $K$, the carrying capacity (between 500 and 2000);
- $x_0 = K/25$, the starting value (4% of $K$).

Use your own ID; the instructor can reproduce your numbers from it. Report $r$, $K$ and $x_0$ in the report's title block.

In [ ]:
student_id = "___"   # type your 9-character student ID (C203.....), in quotes

def params_from_id(identifier):
    key = identifier.strip().upper()
    assert re.fullmatch(r"C203\w{5}", key), ("Type your student ID between the quotes: 9 characters, C203 followed by five more." if key in ("", "___") else f"A Catawba student ID has 9 characters: C203 followed by five more. You typed {len(key)}.")
    h = int(hashlib.sha256(key.encode()).hexdigest(), 16)
    r = 0.30 + (h % 61) / 100               # 0.30, 0.31, ..., 0.90 per day
    K = 500 + 50 * ((h // 61) % 31)         # 500, 550, ..., 2000
    return round(r, 2), K

r, K = params_from_id(student_id)
x0 = K / 25                                  # 4% of K
print(f"student ID: {student_id.strip().upper()}")
print(f"your r  = {r:.2f} per day")
print(f"your K  = {K}")
print(f"your x0 = {x0:g}")

## Part 1. Your model
The right-hand side $f(t,x)=r\,x\,(1-x/K)$ and the exact solution, with your $r$, $K$, $x_0$. The plot shows the exact solution on $[0, 16]$ days.

In [ ]:
def f(t, x):
    return r * x * (1 - x / K)

def exact(t):
    return K * x0 / (x0 + (K - x0) * np.exp(-r * t))

T = 16.0
fine_t = np.linspace(0, T, 400)
plt.figure(figsize=(6.5, 4))
plt.plot(fine_t, exact(fine_t), "-", color=OI[0], linewidth=2, label="exact solution x(t)")
plt.axhline(K, linestyle="--", color=OI[6], label=f"carrying capacity K = {K}")
plt.xlabel("time t (days)")
plt.ylabel("x(t)")
plt.title(f"Your logistic model: r = {r:.2f}, K = {K}, x0 = {x0:g}")
plt.grid(alpha=0.3)
plt.legend(loc="best", fontsize=8)
plt.show()

for t_k in (1, 2, 16):
    print(f"x({t_k}) = {exact(t_k):.3f}")

## Part 2. Hand check (on paper first)
Do these with a calculator **before** running the next cell. One **iteration** is one Euler step $x_{k+1} = x_k + h\,f(t_k, x_k)$. The **absolute error** is $|\text{exact} - \text{Euler}|$.
The slides to use are in the *Step Size and Stability* slides (Module 13): **Why the Logistic Oscillates** ($g'(K) = 1 - hr$, stable for $h < 2/r$) and the worked example on the backup slide **Predict the Safe Step** ($r = 0.8$: $h < 2.5$).

**(a)** Two Euler iterations with $h = 1$ from $x_0$: compute $x_1$ and $x_2$. Compare them with $x(1)$ and $x(2)$ from Part 1 and compute the two absolute errors.

**(b)** Near $K$, linearize: $f'(K) = -r$. For the step map $g(x) = x + h\,f(x)$, $g'(K) = 1 - hr$. Compute **the largest $h$ for which Euler is stable near $K$**.

**(c)** For the three steps $h_A = 1/r$, $h_B = 1.5/r$, $h_C = 2.4/r$ (the cell prints them with 4 decimals), compute $g'(K)$ and **predict** what Euler does near $K$. Choose from: *reaches $K$ and stays*; *settles at $K$ while flipping above and below*; *does not settle (bounces)*.

Then run the cell and compare its values with yours. Put **photos of your hand work** and the matching table in the report's **2. Handwritten Computations**. If a value does not match, find the step that went wrong and note it.

In [ ]:
# The notebook's values: compare with your paper work
x1 = x0 + 1.0 * f(0.0, x0)
x2 = x1 + 1.0 * f(1.0, x1)
err1, err2 = abs(exact(1) - x1), abs(exact(2) - x2)
h_A, h_B, h_C = 1 / r, 1.5 / r, 2.4 / r
gprime = {name: round(1 - h * r, 10) + 0.0 for name, h in (("h_A", h_A), ("h_B", h_B), ("h_C", h_C))}

print(f"{'quantity':<26s} {'notebook value':>14s}")
for name, value, fmt in (("x1 (h = 1)", x1, ".3f"), ("x2 (h = 1)", x2, ".3f"),
                         ("absolute error at t = 1", err1, ".3f"), ("absolute error at t = 2", err2, ".3f"),
                         ("2/r", 2 / r, ".4f"),
                         ("h_A = 1/r", h_A, ".4f"), ("h_B = 1.5/r", h_B, ".4f"), ("h_C = 2.4/r", h_C, ".4f"),
                         ("g'(K) at h_A", gprime["h_A"], ".1f"), ("g'(K) at h_B", gprime["h_B"], ".1f"),
                         ("g'(K) at h_C", gprime["h_C"], ".1f")):
    print(f"{name:<26s} {value:>14{fmt}}")

## Part 3. Order of accuracy (run and observe)
Euler and RK4 run to $T = 16$ with $h = 1, 1/2, \dots, 1/64$, and we measure the absolute error at $T = 16$.
The error behaves like $C\,h^p$, so $\log(\text{error}) = \log C + p\,\log h$: on log–log axes, **the slope is the order $p$**. The plot comes first; the fitted slopes (over the last 4 points, $h = 1/8 \dots 1/64$) are in its legend and printed under it.

In [ ]:
def h_label(k):
    return "1" if k == 0 else f"1/{2**k}"

h_values = np.array([2.0 ** -k for k in range(7)])
err_euler = np.array([abs(exact(T) - euler(f, 0.0, x0, h, T)) for h in h_values])
err_rk4 = np.array([abs(exact(T) - rk4(f, 0.0, x0, h, T)) for h in h_values])
slope_euler = np.polyfit(np.log(h_values[-4:]), np.log(err_euler[-4:]), 1)[0]
slope_rk4 = np.polyfit(np.log(h_values[-4:]), np.log(err_rk4[-4:]), 1)[0]

plt.figure(figsize=(6.5, 4.5))
plt.loglog(h_values, err_euler, "o-", color=OI[0], label=f"Euler (fitted slope {slope_euler:.3f})")
plt.loglog(h_values, err_rk4, "s--", color=OI[1], label=f"RK4 (fitted slope {slope_rk4:.3f})")
plt.loglog(h_values, err_euler[-1] * (h_values / h_values[-1]), ":", color=OI[6], label="reference slope 1")
plt.loglog(h_values, err_rk4[-1] * (h_values / h_values[-1]) ** 4, "-.", color=OI[3], label="reference slope 4")
plt.xlabel("step size h")
plt.ylabel("absolute error at T = 16")
plt.title("Euler and RK4: absolute error against step size")
plt.grid(alpha=0.3, which="both")
plt.legend(loc="best", fontsize=8)
plt.show()

print(f"Fitted slope over the last 4 points (h = 1/8 ... 1/64): Euler {slope_euler:.3f}, RK4 {slope_rk4:.3f}")

The supporting table: **ratio** = (absolute error with step $2h$) / (absolute error with step $h$).

In [ ]:
ratio_euler = err_euler[:-1] / err_euler[1:]
ratio_rk4 = err_rk4[:-1] / err_rk4[1:]
print(f"{'h':>5s} {'iterations n':>13s} {'Euler absolute error':>21s} {'ratio':>7s} {'RK4 absolute error':>19s} {'ratio':>7s}")
for i, h in enumerate(h_values):
    r_e = "-" if i == 0 else f"{ratio_euler[i - 1]:.3f}"
    r_r = "-" if i == 0 else f"{ratio_rk4[i - 1]:.2f}"
    print(f"{h_label(i):>5s} {n_steps(T, h):>13d} {err_euler[i]:>21.4g} {r_e:>7s} {err_rk4[i]:>19.4g} {r_r:>7s}")

**Observation.** What are the two fitted slopes, and what happens to each method's absolute error when $h$ is halved? Look at all the rows of the table.

*Your observation:*

## Part 4. Same cost (run and observe)
The cost of a method is the number of **evaluations of $f$**. Euler uses one per iteration and RK4 uses four. The cell counts them for two pairs of runs that cost the same.

In [ ]:
calls = 0
def f_counted(t, x):
    global calls
    calls += 1
    return f(t, x)

same_cost = []
print(f"{'method':>6s} {'evaluations of f':>17s} {'h':>5s} {'absolute error':>15s}")
for name, method, k in (("Euler", euler, 4), ("RK4", rk4, 2), ("Euler", euler, 6), ("RK4", rk4, 4)):
    calls = 0
    x_end = method(f_counted, 0.0, x0, 2.0 ** -k, T)
    error = abs(exact(T) - x_end)
    same_cost.append((name, calls, 2.0 ** -k, error))
    print(f"{name:>6s} {calls:>17d} {h_label(k):>5s} {error:>15.4g}")

**Observation.** For the same number of evaluations of $f$, how do the two methods compare?

*Your observation:*

## Part 5. Testing the stability prediction (run and observe)
Euler and RK4 with $h_A$, $h_B$, $h_C$ for **60 iterations** from $x_0$. The panels show $x_k$ against the iteration $k$, with $K$ dashed. The table shows the last four values and whether the method **settles at $K$** ($|x_k - K| < 0.01\,K$ for the last 10 iterations).

Compare with your prediction in Part 2(c).

In [ ]:
# Here the number of iterations (60) is fixed, not the final time T: each h reaches a different time 60h,
# and what we watch is how the iterates behave near K. So we call euler and rk4 for one iteration at a time
# (from t to t + h) and keep every value.
N_ITER = 60

def iterates(method, h, n=N_ITER):
    xs = [x0]
    t = 0.0
    for k in range(n):
        xs.append(method(f, t, xs[-1], h, t + h))
        t = t + h
    return np.array(xs)

def settles(xs):
    return bool(np.all(np.abs(xs[-10:] - K) < 0.01 * K))

stability = []
fig, axes = plt.subplots(1, 3, figsize=(12, 3.8), sharey=True)
for ax, (name, h) in zip(axes, (("h_A", h_A), ("h_B", h_B), ("h_C", h_C))):
    xe, xr = iterates(euler, h), iterates(rk4, h)
    stability.append((name, h, h * r, round(1 - h * r, 10) + 0.0, xe[-4:], xr[-4:], settles(xe), settles(xr)))
    k = np.arange(N_ITER + 1)
    ax.plot(k, xe, "o-", color=OI[0], markersize=3, label="Euler")
    ax.plot(k, xr, "s:", color=OI[1], markersize=3, label="RK4")
    ax.axhline(K, linestyle="--", color=OI[6], label=f"K = {K}")
    ax.set_title(f"{name} = {h:.4f}  (hr = {h * r:.1f})")
    ax.set_xlabel("iteration k")
    ax.grid(alpha=0.3)
    ax.legend(loc="lower right", fontsize=8)
axes[0].set_ylabel("x_k")
fig.suptitle("Euler and RK4 near K for three step sizes")
plt.tight_layout()
plt.show()

GP = "g'(K)"
print(f"{'h':>12s} {'hr':>4s} {GP:>10s}  {'Euler: last four values':<34s} {'RK4: last four values':<34s} {'Euler settles?':>14s} {'RK4 settles?':>12s}")
for name, h, hr, g, le, lr, se, sr in stability:
    print(f"{name + ' = ' + format(h, '.4f'):>12s} {hr:>4.1f} {g:>10.1f}  {', '.join(f'{v:.2f}' for v in le):<34s} "
          f"{', '.join(f'{v:.2f}' for v in lr):<34s} {'yes' if se else 'no':>14s} {'yes' if sr else 'no':>12s}")

**Observation.** Did Euler and RK4 do what you predicted in Part 2(c) for $h_A$, $h_B$, $h_C$?

*Your observation:*

## Part 6. Your results for the report (given)
Copy these lines into the report: the parameters go in the title block, and the slopes and both tables go in **1. Results**. The tables are Markdown and paste into Word as text that you can convert to a table. The "predicted" column is for your own Part 2 prediction.

In [ ]:
print(f"Personal parameters: r = {r:.2f}, K = {K}, x0 = {x0:g}\n")
print(f"Fitted slopes (last 4 points, h = 1/8 ... 1/64): Euler {slope_euler:.3f}, RK4 {slope_rk4:.3f}\n")
print("(1) Order of accuracy, absolute error at T = 16")
print("| h | iterations n | Euler absolute error | ratio | RK4 absolute error | ratio |")
print("|---|---|---|---|---|---|")
for i, h in enumerate(h_values):
    r_e = "-" if i == 0 else f"{ratio_euler[i - 1]:.3f}"
    r_r = "-" if i == 0 else f"{ratio_rk4[i - 1]:.2f}"
    print(f"| {h_label(i)} | {n_steps(T, h)} | {err_euler[i]:.4g} | {r_e} | {err_rk4[i]:.4g} | {r_r} |")
print("\n(2) Stability near K, 60 iterations")
print("| h | g'(K) | predicted | Euler settles? | RK4 settles? |")
print("|---|---|---|---|---|")
for name, h, hr, g, le, lr, se, sr in stability:
    print(f"| {name} = {h:.4f} | {g:.1f} | see your Part 2 | {'yes' if se else 'no'} | {'yes' if sr else 'no'} |")

## Part 7 (Challenge Project, skip now)
In the Challenge Project you will reuse `euler` and `rk4` on the SIR model, fit it to outbreak data, and choose a solver and a step. Keep this notebook.

## Before you submit the report
- Run **Kernel → Restart & Run All** (Colab: *Runtime → Restart and run all*) so every output is fresh, then save.
- Check that the numbers in your report match this notebook.
- Submit your report (`Lastname_GP_Testing_Solver_Report.docx`). Keep this notebook: the Challenge Project builds on it.